#ДЗ.8 Детектор большого объема

Построить модель детектора со следующими параметрами:
R = 15 метров
N = 2000
r = 10 см
a = 0.5
Аналогично задаче 1, расположить фотоумножители достаточно равномерно.
Оптимизировать алгоритм для поиска попадания фотона на фотоумножитель. Алгоритм должен работать быстрее, чем итерирование по полному списку фотоумножителей.

чтобы  расположить фотоумножители достаточно равномерн, расположим фотоумножители на сорока параллелях и количество точек на каждой параллели выберем пропорционально ее радиусу


In [26]:
import numpy as np

class Detector:
    def __init__(self, R: float, N: int, r: float, a: float):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

    def dots_on_parallels(self): 
        n_theta = 40

        cos_theta = 1 - 2*(np.arange(n_theta) + 0.5)/n_theta
        theta = np.arccos(cos_theta)

        n_phi = np.round(self.N * np.sin(theta) / np.sum(np.sin(theta))).astype(int) #распределяем 2000 фотоумножителей между параллелями пропорционально \(\sin\theta\)

        # после округления сумма может немного отличаться от 2000, поэтому корректируем число точек
        n_phi[n_theta//2] += self.N - np.sum(n_phi)

        dots = []

        for i in range(n_theta):
            phi = np.linspace(0, 2*np.pi, n_phi[i], endpoint=False)

            for angle in phi:
                x = self.R * np.sin(theta[i]) * np.cos(angle)
                y = self.R * np.sin(theta[i]) * np.sin(angle)
                z = self.R * np.cos(theta[i])

                dots.append([x, y, z])

        self.photocathode_centers = np.array(dots)
        
        # Сортируем центры фотокатодов по координате z
        order = np.argsort(self.photocathode_centers[:, 2])

        self.sorted_centers = self.photocathode_centers[order]
        self.sorted_z = self.sorted_centers[:, 2]


        #новый способ
    def new_method(self, coordinats):
        # Отбираем центры фотокатодов, близкие к точке по координате z

        left = np.searchsorted(self.sorted_z, coordinats[2] - self.r)
        right = np.searchsorted(self.sorted_z, coordinats[2] + self.r, side='right')

        centers = self.sorted_centers[left:right]

        # Проверяем расстояния только до выбранных центров
        lengths = np.linalg.norm(centers - coordinats, axis=1)

        return np.any(lengths <= self.r)

        #из task_1
    def is_on_photocathode(self, coordinats):
        
        lengths = np.linalg.norm(self.photocathode_centers - coordinats, axis=1)

        return np.any(lengths <= self.r)

detector = Detector(R=15.0, N=2000, r=0.1, a=0.5)
detector.dots_on_parallels()


In [27]:
# проверка
print(detector.photocathode_centers.shape)

lengths = np.linalg.norm(detector.photocathode_centers, axis=1)
assert np.allclose(lengths, detector.R)

print("Нет ошибок")

(2000, 3)
Нет ошибок


In [28]:
import time

# Генерируем 5000 случайных точек на поверхности сферы
coordinats = np.random.normal(size=(5000, 3))
coordinats = detector.R * coordinats / np.linalg.norm(coordinats, axis=1)[:, None]

# Старый способ
start = time.perf_counter()

old_results = []

for point in coordinats:
    old_results.append(detector.is_on_photocathode(point))

time_old = time.perf_counter() - start

# Новый способ
start = time.perf_counter()

new_results = []

for point in coordinats:
    new_results.append(detector.new_method(point))

time_new = time.perf_counter() - start

# Проверяем, что результаты совпадают
assert np.array_equal(old_results, new_results)

print("Время (старый способ):", time_old)
print("Время (новый способ):", time_new)
print("Ускорение в", round(time_old / time_new, 2), "раза")
print("Проверка пройдена")


Время (старый способ): 0.3338502999395132
Время (новый способ): 0.10166590008884668
Ускорение в 3.28 раза
Проверка пройдена


Для ускорения поиска попаданий центры фотоумножителей были отсортированы по координате z, чтобы проверять расстояния только до части фотоумножителей расположенных рядом с точкой попадания.
При проверке 5000 случайных точек время работы уменьшилось с 0.334 с до 0.102 с, то есть новый метод оказался примерно в 3.28 раза быстрее. Результаты обоих методов совпали.